In [1]:
import numpy as np
import pandas as pd
from haversine import haversine_vector, Unit

In [2]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"

In [3]:
df_truck_path = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_POI_Data - Copy.csv")
df_merged_file = pd.read_csv(
    path + r"4. Working Data Files\Traffic Files\Capstone_truck\merged_filtered_file_11_14.csv")

C:\Users\bhavy\AppData\Local\Temp\ipykernel_27928\967807618.py:2: DtypeWarning: Columns (2,87) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged_file = pd.read_csv(path + r"4. Working Data Files\Traffic Files\Capstone_truck\merged_filtered_file_11_14.csv")


In [196]:
park_data_1 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_1 - Copy.csv")
park_data_2 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_2 - Copy.csv")
park_data_3 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_3 - Copy.csv")
park_data = pd.concat([park_data_1, park_data_2, park_data_3], ignore_index=True)

In [198]:
park_data[park_data["pin id"] == "dd1016d687d5960a8f279198a94d0cc5"]

,pinname,parking status,time(utc),pinlat,pinlon,pin id,object,city,route_num
130338,"Newborn TruckStop, Tallapoosa GA",Full,2025/04/08 05:28:23,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
327444,"Newborn TruckStop, Tallapoosa GA",Full,2025/04/08 04:42:23,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
327445,"Newborn TruckStop, Tallapoosa GA",Lots,2025/04/08 03:44:07,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
327446,"Newborn TruckStop, Tallapoosa GA",Lots,2025/04/08 02:58:07,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
327447,"Newborn TruckStop, Tallapoosa GA",Full,2025/04/08 02:20:05,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
...,...,...,...,...,...,...,...,...,...
483461,"Newborn TruckStop, Tallapoosa GA",Lots,2025/04/08 16:50:51,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
483462,"Newborn TruckStop, Tallapoosa GA",Lots,2025/04/08 12:08:28,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
483463,"Newborn TruckStop, Tallapoosa GA",Lots,2025/04/08 11:51:28,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20
483464,"Newborn TruckStop, Tallapoosa GA",Full,2025/04/08 09:27:29,33.681568,-85.264077,dd1016d687d5960a8f279198a94d0cc5,Truck Stops,Tallapoosa,20


In [5]:
df_truck_path.head(2)

,truckParkingSpotCount,showerCount,location,overnightParking,address,Pin ID,phone,state,laundry,lat,lng
0,312.0,9.0,"[-86.87248630306287, 34.050601881119654]",True,426 AL-69,c81e728d9d4c2f636f067f89cc14862c,12562871299,AL,True,34.050602,-86.872486
1,50.0,6.0,"[-85.95542833862305, 32.405957794002745]",True,428 Main St,a87ff679a2f3e71d9181a67b7542122c,1(334)727-3354,AL,True,32.405958,-85.955428


In [6]:
df_merged_file.columns

Index(['datayear', 'stateid', 'routeid', 'beginpoint', 'endpoint', 'f_system',
       'facility_type', 'urban_id', 'through_lanes', 'aadt', 'aadt_date',
       'is_restricted', 'nhs', 'county_id', 'ownership',
       'maintenance_operations', 'managed_lanes', 'managed_lanes_type',
       'peak_lanes', 'counter_peak_lanes', 'toll_id', 'lane_width',
       'median_type', 'median_width', 'shoulder_type', 'shoulder_width_r',
       'shouder_width_l', 'peak_parking', 'dir_through_lanes', 'turn_lanes_r',
       'turn_lanes_l', 'signal_type', 'pct_green_time', 'number_signals',
       'stop_signs', 'at_grade_other', 'aadt_single_unit', 'aadt_combination',
       'pct_dh_single_unit', 'pct_dh_combination', 'k_factor', 'dir_factor',
       'future_aadt', 'future_aadt_year', 'access_control', 'speed_limit',
       'iri', 'iri_date', 'psr', 'surface_type', 'rutting', 'faulting',
       'cracking_percent', 'year_last_improvement', 'year_last_construction',
       'last_overlay_thickness', 'thickne

In [7]:
df_truck_path = df_truck_path[["Pin ID", "lat", "lng", "truckParkingSpotCount"]].drop_duplicates()
df_merged_file = df_merged_file[["routeid", "beginpoint", "endpoint", "f_system", "MID_LAT",
                                 "MID_LONG"]].drop_duplicates()

In [8]:
df_merged_file = df_merged_file[~df_merged_file["MID_LAT"].isnull()].copy()

In [9]:
df_stop = pd.merge(df_truck_path, df_merged_file, how="cross")

In [10]:
df_truck_path.shape[0] * df_merged_file.shape[0]

236604524

In [11]:
df_stop['distance_miles'] = haversine_vector(df_stop[['lat', 'lng']],
                                             df_stop[['MID_LAT', 'MID_LONG']],
                                             Unit.MILES
                                             ) * 1.16

In [12]:
df_stop

,Pin ID,lat,lng,truckParkingSpotCount,routeid,beginpoint,endpoint,f_system,MID_LAT,MID_LONG,distance_miles
0,c81e728d9d4c2f636f067f89cc14862c,34.050602,-86.872486,312.0,AL0000120000,21.789,21.810,3,31.751882,-88.114977,202.317830
1,c81e728d9d4c2f636f067f89cc14862c,34.050602,-86.872486,312.0,IN0005650000,11.979,12.026,1,34.689744,-86.716074,52.260786
2,c81e728d9d4c2f636f067f89cc14862c,34.050602,-86.872486,312.0,AL0000120000,127.826,127.828,3,31.346290,-86.525873,218.002763
3,c81e728d9d4c2f636f067f89cc14862c,34.050602,-86.872486,312.0,AL0000121000,186.625,189.521,3,31.302009,-85.767341,232.564612
4,c81e728d9d4c2f636f067f89cc14862c,34.050602,-86.872486,312.0,IN0000100000,21.866,21.930,1,30.637412,-88.094546,285.795314
...,...,...,...,...,...,...,...,...,...,...,...
236604519,fdc899369c3511966cedb8473e1782fb,32.424707,-81.791215,10.0,92471000,37.100,37.186,2,28.300524,-81.358629,331.896313
236604520,fdc899369c3511966cedb8473e1782fb,32.424707,-81.791215,10.0,90020000,9.959,9.965,3,24.625492,-81.598969,625.240470
236604521,fdc899369c3511966cedb8473e1782fb,32.424707,-81.791215,10.0,16060000,9.400,9.500,3,28.015258,-81.915309,353.514517
236604522,fdc899369c3511966cedb8473e1782fb,32.424707,-81.791215,10.0,10075000,28.800,28.900,1,28.025302,-82.338011,354.630609


In [13]:
idx = df_stop.groupby('Pin ID')['distance_miles'].idxmin()
nearest = df_stop.loc[idx].reset_index()

In [14]:
nearest = nearest[
    ["Pin ID", "lat", "lng", "truckParkingSpotCount", "f_system", "routeid", "beginpoint", "endpoint"]].copy()

In [15]:
nearest

,Pin ID,lat,lng,truckParkingSpotCount,f_system,routeid,beginpoint,endpoint
0,00038787e666dbf59f419128a0ee3a66,30.406303,-81.743304,97.0,3,72080000,7.600,7.700
1,0007cda84fafdcf42f96c4f4adb7f8ce,28.529926,-81.731362,0.0,3,11200000,13.400,13.500
2,005b00cc243fb4f39296fa0f16a21482,30.612343,-86.978559,50.0,1,58002000,13.700,13.800
3,0064203b17d778528816f80ef1e29208,30.766060,-85.686060,NaN,1,52002000,17.100,17.129
4,00c23232a9ac7754e1c515d4b6a63f2e,28.799962,-81.274442,0.0,3,77010000,12.500,12.600
...,...,...,...,...,...,...,...,...
1607,fecf8417ed709b82623dde69c75c46bc,34.716627,-87.688432,NaN,3,AL0000020000,24.898,24.931
1608,fee03b7e1be695f06ffdc8203f79e52a,33.878155,-84.734239,NaN,1,1000100040200INC,39.376,39.476
1609,ff1ced3097ccf17c1e67506cdad9ac95,32.379220,-82.061574,20.0,1,1000100040400INC,103.446,103.546
1610,ff39ad2009df462df3dc2ec010c1add8,26.147443,-80.629998,NaN,1,86075000,29.700,29.800


In [16]:
stop_name_df = park_data[["pinname", "pin id"]].drop_duplicates()
stop_name_df

,pinname,pin id
0,Rest Area WB,005b00cc243fb4f39296fa0f16a21482
512,Friendly Express #31,f312db0543cc671be332446ebb76d2e9
633,Rest Area EB,41dbaeec4466f58cd76a81b7add4a279
1621,Rest Area WB,98c2473648afc991669f9b9334c11072
1652,Spartanburg Walmart Supercenter Store #1035,0bc10d8a74dbafbf242e30433e83aa56
...,...,...
1759820,ONE9 Dealer (One9 Fuel Network) #1365,928e734545f99a375312f4a20606f560
1760145,McDonald's,d3fdb21345d8171619904e061ad625c2
1760229,Marathon #102,665d5cbb82b5785d9f344c46417c6c36
1761273,"Truck Parking Club - Crawfordville, FL Truck a...",95d44ae4e547e2d248598628cb571fc8


In [158]:
print(nearest.shape)
stop_tab_df = pd.merge(nearest, stop_name_df, left_on="Pin ID", right_on="pin id", how="left")
print(stop_tab_df.shape)


(1612, 8)
(1612, 10)


In [159]:
stop_tab_df["link_id"] = stop_tab_df["routeid"].astype("str") + "_" + stop_tab_df["beginpoint"].astype("str") + "_" + \
                         stop_tab_df["endpoint"].astype("str")
stop_tab_df = stop_tab_df[["pin id", "pinname", "lat", "lng", "truckParkingSpotCount", "f_system", "link_id"]].copy()
stop_tab_df["review_score"] = ""
stop_tab_df["amenities_score"] = ""
stop_tab_df

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score
0,00038787e666dbf59f419128a0ee3a66,Truck Spot Properties,30.406303,-81.743304,97.0,3,72080000_7.6_7.7,,
1,0007cda84fafdcf42f96c4f4adb7f8ce,Clermont Walmart Supercenter Store #2695,28.529926,-81.731362,0.0,3,11200000_13.4_13.5,,
2,005b00cc243fb4f39296fa0f16a21482,Rest Area WB,30.612343,-86.978559,50.0,1,58002000_13.7_13.8,,
3,0064203b17d778528816f80ef1e29208,"Truck Parking Club - Bonifay, FL Truck & Trail...",30.766060,-85.686060,NaN,1,52002000_17.1_17.129,,
4,00c23232a9ac7754e1c515d4b6a63f2e,SFG Farmers Market,28.799962,-81.274442,0.0,3,77010000_12.5_12.6,,
...,...,...,...,...,...,...,...,...,...
1607,fecf8417ed709b82623dde69c75c46bc,Tuscumbia Texaco,34.716627,-87.688432,NaN,3,AL0000020000_24.898_24.931,,
1608,fee03b7e1be695f06ffdc8203f79e52a,Affordable Parking,33.878155,-84.734239,NaN,1,1000100040200INC_39.376_39.476,,
1609,ff1ced3097ccf17c1e67506cdad9ac95,bp,32.379220,-82.061574,20.0,1,1000100040400INC_103.446_103.546,,
1610,ff39ad2009df462df3dc2ec010c1add8,Rest area,26.147443,-80.629998,NaN,1,86075000_29.7_29.8,,


In [160]:
stop_tab_df[stop_tab_df["pin id"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score
1090,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,1,1000100040200INC_4.5_4.6,,


# Ginny & Christina data check

In [161]:
df_comb = pd.read_csv(path + r"4. Working Data Files\TruckStopsCombined.csv")
df_comb = df_comb[["StoreNumber", "Latitude", "Longitude", "ParkingSpaces"]].drop_duplicates()

In [162]:
df_comb = pd.merge(stop_tab_df, df_comb, how="cross")

In [163]:
df_comb['distance_miles'] = haversine_vector(df_comb[['lat', 'lng']],
                                             df_comb[['Latitude', 'Longitude']],
                                             Unit.MILES
                                             ) * 1.16

In [164]:
idx = df_comb.groupby('pin id')['distance_miles'].idxmin()
df_comb_nearest = df_comb.loc[idx].reset_index()

In [165]:
# df_comb_nearest.to_excel("1.xlsx")

In [166]:
gc_tp_df = df_comb_nearest.copy()

gc_tp_df = gc_tp_df[gc_tp_df["distance_miles"] < .1].copy()
gc_tp_df_gp = gc_tp_df.groupby(["StoreNumber"]).agg({"Latitude": "count"}).reset_index()
gc_tp_df_gp.rename(columns={"Latitude": "count_store"}, inplace=True)
gc_tp_df = gc_tp_df[(gc_tp_df["truckParkingSpotCount"] != 0)].copy()
gc_tp_df = gc_tp_df[(~gc_tp_df["truckParkingSpotCount"].isnull())].copy()
gc_tp_df = gc_tp_df[gc_tp_df["truckParkingSpotCount"] != gc_tp_df["ParkingSpaces"]].copy()
gc_tp_df = pd.merge(gc_tp_df, gc_tp_df_gp, on="StoreNumber", how="left")
gc_tp_df = gc_tp_df[gc_tp_df["count_store"] == 1].copy()
gc_tp_df["truckParkingSpotCount"] = gc_tp_df["ParkingSpaces"]
gc_tp_df = gc_tp_df[["pin id", 'truckParkingSpotCount']].copy()
gc_tp_df.rename(columns={"truckParkingSpotCount": "ucount"}, inplace=True)

In [167]:
stop_tab_df = pd.merge(stop_tab_df, gc_tp_df, on="pin id", how="left")

In [168]:
stop_tab_df[stop_tab_df["pin id"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount
1090,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,1,1000100040200INC_4.5_4.6,,,NaN


In [169]:
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["ucount"].isnull(), stop_tab_df["truckParkingSpotCount"],
                                                stop_tab_df["ucount"])

In [170]:
stop_tab_df[stop_tab_df["pin id"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount
1090,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,1,1000100040200INC_4.5_4.6,,,NaN


In [171]:
stop_tab_df

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount
0,00038787e666dbf59f419128a0ee3a66,Truck Spot Properties,30.406303,-81.743304,97.0,3,72080000_7.6_7.7,,,NaN
1,0007cda84fafdcf42f96c4f4adb7f8ce,Clermont Walmart Supercenter Store #2695,28.529926,-81.731362,0.0,3,11200000_13.4_13.5,,,NaN
2,005b00cc243fb4f39296fa0f16a21482,Rest Area WB,30.612343,-86.978559,50.0,1,58002000_13.7_13.8,,,NaN
3,0064203b17d778528816f80ef1e29208,"Truck Parking Club - Bonifay, FL Truck & Trail...",30.766060,-85.686060,NaN,1,52002000_17.1_17.129,,,NaN
4,00c23232a9ac7754e1c515d4b6a63f2e,SFG Farmers Market,28.799962,-81.274442,0.0,3,77010000_12.5_12.6,,,NaN
...,...,...,...,...,...,...,...,...,...,...
1607,fecf8417ed709b82623dde69c75c46bc,Tuscumbia Texaco,34.716627,-87.688432,NaN,3,AL0000020000_24.898_24.931,,,NaN
1608,fee03b7e1be695f06ffdc8203f79e52a,Affordable Parking,33.878155,-84.734239,NaN,1,1000100040200INC_39.376_39.476,,,NaN
1609,ff1ced3097ccf17c1e67506cdad9ac95,bp,32.379220,-82.061574,20.0,1,1000100040400INC_103.446_103.546,,,NaN
1610,ff39ad2009df462df3dc2ec010c1add8,Rest area,26.147443,-80.629998,NaN,1,86075000_29.7_29.8,,,NaN


### Intra Chris duplicates

In [172]:
stop_tab_df[stop_tab_df["pin id"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount
1090,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,1,1000100040200INC_4.5_4.6,,,NaN


In [173]:
a = stop_tab_df[['pin id', 'pinname', 'lat', 'lng', 'truckParkingSpotCount']].drop_duplicates().copy()
a = pd.merge(a, a, how="cross")
a['distance_miles'] = haversine_vector(a[['lat_x', 'lng_x']],
                                       a[['lat_y', 'lng_y']],
                                       Unit.MILES
                                       ) * 1.16
a["pair"] = a.apply(lambda x: tuple(sorted([x["pin id_x"], x["pin id_y"]])), axis=1)
a = a.drop_duplicates(subset="pair", keep="first").drop(columns=["pair"])

In [174]:
a[a["pin id_x"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id_x,pinname_x,lat_x,lng_x,truckParkingSpotCount_x,pin id_y,pinname_y,lat_y,lng_y,truckParkingSpotCount_y,distance_miles
1758170,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,0.000000
1758171,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,aefccd61388dd113ae45210bb0f5c7db,Sunoco,33.580472,-86.025531,5.0,51.497166
1758172,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,aefe32daa6c57fd474affa76285ee12c,Rest Area,32.937465,-81.517438,1.0,257.915042
1758173,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,af07b12378c57ffdb56a391f4c4e4fa8,Circle K,34.247184,-85.161058,NaN,45.765464
1758174,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,afa58517440a9cdacab7d6a0174a376b,Marathon,34.479025,-85.712258,15.0,70.440912
...,...,...,...,...,...,...,...,...,...,...,...
1758687,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,fecf8417ed709b82623dde69c75c46bc,Tuscumbia Texaco,34.716627,-87.688432,NaN,180.839573
1758688,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,fee03b7e1be695f06ffdc8203f79e52a,Affordable Parking,33.878155,-84.734239,NaN,38.594797
1758689,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,ff1ced3097ccf17c1e67506cdad9ac95,bp,32.379220,-82.061574,20.0,239.161184
1758690,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,ff39ad2009df462df3dc2ec010c1add8,Rest area,26.147443,-80.629998,NaN,684.129902


In [175]:
a = a[a["distance_miles"] < .1].copy()
a = a[a["pin id_x"] != a["pin id_y"]].copy()
a.dropna(inplace=True)
a = a[a["truckParkingSpotCount_x"] != 0].copy()
a = a[a["truckParkingSpotCount_y"] != 0].copy()
print(a.shape)
# TODO: Discuss with @Sam
list_to_rm = a[a["truckParkingSpotCount_y"] == a["truckParkingSpotCount_x"]]["pin id_y"].unique()
a = a[a["truckParkingSpotCount_y"] != a["truckParkingSpotCount_x"]].copy()
print(a.shape)
a['bigger'] = np.where(a["truckParkingSpotCount_y"] > a["truckParkingSpotCount_x"], "y", "x")
a["truckParkingSpotCount_y"] = np.where(a['bigger'] == "x", 0, a["truckParkingSpotCount_y"])
a["truckParkingSpotCount_x"] = np.where(a['bigger'] == "y", 0, a["truckParkingSpotCount_x"])
a

(44, 11)
(40, 11)


,pin id_x,pinname_x,lat_x,lng_x,truckParkingSpotCount_x,pin id_y,pinname_y,lat_y,lng_y,truckParkingSpotCount_y,distance_miles,bigger
188120,0f7dfe5e8787efa0406adbb556d76c4a,Rest Area NB,31.008825,-85.407196,0.0,b59978537754917c2df87b96ac795ef2,Rest Area SB,31.008808,-85.407251,8.0,0.004034,y
219491,116e195af1954fe5253cf0a55f07ef48,Rest Area SB,26.549514,-81.792156,0.0,2807e5da47a64f8e7f640390a55efc9d,Lee County Rest Area,26.549569,-81.792195,40.0,0.005204,y
219635,116e195af1954fe5253cf0a55f07ef48,Rest Area SB,26.549514,-81.792156,0.0,4490759a6b69ecb7c170d8e1476be569,Lee County Rest Area,26.549272,-81.792249,20.0,0.020531,y
287288,1961aadca386eb5de3a75111cdffcbed,Manatee County Rest Area,27.584565,-82.614143,0.0,3bc188adb791a56ccce03630b0d9f593,Rest Area NB,27.584546,-82.614169,16.0,0.002426,y
308941,1b829e73430744463fbf634480db5862,GHANSHYAM FOOD LLC,32.405940,-85.956520,0.0,a87ff679a2f3e71d9181a67b7542122c,Petro Shorter #348,32.405958,-85.955428,50.0,0.073883,y
378046,23363e9a5104d797987048558121b51f,"IdleAir at Newnan, GA",33.320596,-84.778316,0.0,887a185b1a4080193d5cf63873ac6d70,Pilot Travel Center #422,33.321115,-84.777822,95.0,0.053154,y
417911,2807e5da47a64f8e7f640390a55efc9d,Lee County Rest Area,26.549569,-81.792195,40.0,4490759a6b69ecb7c170d8e1476be569,Lee County Rest Area,26.549272,-81.792249,0.0,0.024155,x
437428,294df9cc7cd9c1808a90d48e3390a94f,Rest Area NB SB,28.787004,-81.982319,28.0,5d55e7c13b0f4d7cf9d5d55d3af329c8,Okahumpka Service Plaza NB SB,28.786862,-81.982424,0.0,0.013557,x
551705,3990fefd135fe61aeea3fa3feed9faaa,Gov. Guy Hunt Rest Stop,34.074060,-86.863667,15.0,4421113f38b3ce3467c194394e8ca46c,Rest Area NB,34.074940,-86.864551,0.0,0.091731,x
636218,42a19e8a17c02dfa58c634ac57a1037f,Chevron,33.681554,-85.263703,0.0,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,0.079799,y


In [176]:
# a.to_excel("Intra Chris comp.xlsx", index=False)

In [177]:
a[a["pin id_y"] == "c4f7bb32268a2d67ce744a6fc57042d6"]

,pin id_x,pinname_x,lat_x,lng_x,truckParkingSpotCount_x,pin id_y,pinname_y,lat_y,lng_y,truckParkingSpotCount_y,distance_miles,bigger
1621285,a1313bec66a803f38868ce8d864719f6,Godfather's Pizza Express,34.227471,-83.499254,0.0,c4f7bb32268a2d67ce744a6fc57042d6,Love's Travel Stop # 765,34.227399,-83.499387,114.0,0.010531,y
1792157,b32e8760418e68f23c811a1cfd6bda78,AmBest,34.227798,-83.498901,165.0,c4f7bb32268a2d67ce744a6fc57042d6,Love's Travel Stop # 765,34.227399,-83.499387,0.0,0.045386,x


In [178]:
id_x = a[["pin id_x", "truckParkingSpotCount_x"]].drop_duplicates().copy()
id_y = a[["pin id_y", "truckParkingSpotCount_y"]].drop_duplicates().copy()
id_x.rename(columns={"pin id_x": "pin id", "truckParkingSpotCount_x": "truckParkingSpotCount"}, inplace=True)
id_y.rename(columns={"pin id_y": "pin id", "truckParkingSpotCount_y": "truckParkingSpotCount"}, inplace=True)
unique_id = pd.concat([id_x, id_y], ignore_index=True)

In [179]:
unique_id.drop_duplicates(inplace=True)
unique_id.rename(columns={"truckParkingSpotCount": "ucount2"}, inplace=True)

In [180]:
a = unique_id.groupby(["pin id"]).agg({"ucount2": "count"}).reset_index()
unique_id[unique_id["pin id"].isin(a[a["ucount2"] > 1]["pin id"].unique())]

,pin id,ucount2
27,aeaeb80c3d5feeb0e2c2c7b0938045c2,0.0
31,c063c0736a23234b286331c4df438050,5.0
33,c85116ddf63252b9b45f31d7f4938681,20.0
37,4490759a6b69ecb7c170d8e1476be569,20.0
41,4490759a6b69ecb7c170d8e1476be569,0.0
44,aeaeb80c3d5feeb0e2c2c7b0938045c2,90.0
56,c85116ddf63252b9b45f31d7f4938681,0.0
62,c4f7bb32268a2d67ce744a6fc57042d6,114.0
65,c063c0736a23234b286331c4df438050,0.0
67,c4f7bb32268a2d67ce744a6fc57042d6,0.0


In [181]:
print(stop_tab_df.shape)
stop_tab_df = pd.merge(stop_tab_df, unique_id, on="pin id", how="left")
print(stop_tab_df.shape)
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["ucount2"].isnull(), stop_tab_df["truckParkingSpotCount"],
                                                stop_tab_df["ucount2"])

(1612, 10)
(1617, 11)


In [182]:
stop_tab_df[stop_tab_df["pin id"] == "aeaeb80c3d5feeb0e2c2c7b0938045c2"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount,ucount2
1091,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,0.0,1,1000100040200INC_4.5_4.6,,,NaN,0.0
1092,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,90.0,1,1000100040200INC_4.5_4.6,,,NaN,90.0


In [183]:
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["pin id"].isin(list_to_rm), 0,
                                                stop_tab_df["truckParkingSpotCount"])

In [184]:
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["pinname"] == "ONE9 Dealer (One9 Fuel Network) #1365", 60,
                                                stop_tab_df["truckParkingSpotCount"])

In [185]:
stop_tab_df[stop_tab_df["pinname"] == "ONE9 Dealer (One9 Fuel Network) #1365"]

,pin id,pinname,lat,lng,truckParkingSpotCount,f_system,link_id,review_score,amenities_score,ucount,ucount2
911,928e734545f99a375312f4a20606f560,ONE9 Dealer (One9 Fuel Network) #1365,32.744745,-85.279124,60.0,1,IN0000850000_70.576_70.595,,,NaN,NaN


In [186]:
stop_tab_df = stop_tab_df[stop_tab_df["truckParkingSpotCount"] != 0].copy()
stop_tab_df = stop_tab_df[~stop_tab_df["truckParkingSpotCount"].isnull()]

In [187]:
a = stop_tab_df[['pin id', 'pinname', 'lat', 'lng', 'truckParkingSpotCount']].drop_duplicates().copy()
a = pd.merge(a, a, how="cross")
a['distance_miles'] = haversine_vector(a[['lat_x', 'lng_x']],
                                       a[['lat_y', 'lng_y']],
                                       Unit.MILES
                                       ) * 1.16
a["pair"] = a.apply(lambda x: tuple(sorted([x["pin id_x"], x["pin id_y"]])), axis=1)
a = a.drop_duplicates(subset="pair", keep="first").drop(columns=["pair"])

In [188]:
a = a[a["distance_miles"] < .1].copy()
a = a[a["pin id_x"] != a["pin id_y"]].copy()
a.dropna(inplace=True)
a = a[a["truckParkingSpotCount_x"] != 0].copy()
a = a[a["truckParkingSpotCount_y"] != 0].copy()
print(a.shape)
# TODO: Discuss with @Sam
list_to_rm = a[a["truckParkingSpotCount_y"] == a["truckParkingSpotCount_x"]]["pin id_y"].unique()
a = a[a["truckParkingSpotCount_y"] != a["truckParkingSpotCount_x"]].copy()
print(a.shape)
a['bigger'] = np.where(a["truckParkingSpotCount_y"] > a["truckParkingSpotCount_x"], "y", "x")
a["truckParkingSpotCount_y"] = np.where(a['bigger'] == "x", 0, a["truckParkingSpotCount_y"])
a["truckParkingSpotCount_x"] = np.where(a['bigger'] == "y", 0, a["truckParkingSpotCount_x"])
a

(5, 11)
(5, 11)


,pin id_x,pinname_x,lat_x,lng_x,truckParkingSpotCount_x,pin id_y,pinname_y,lat_y,lng_y,truckParkingSpotCount_y,distance_miles,bigger
93302,2807e5da47a64f8e7f640390a55efc9d,Lee County Rest Area,26.549569,-81.792195,40.0,4490759a6b69ecb7c170d8e1476be569,Lee County Rest Area,26.549272,-81.792249,0.0,0.024155,x
338310,8efa9015a4ef4632a954e820eca834ad,I-20 TRUCK STOP,33.589529,-86.163795,100.0,c85116ddf63252b9b45f31d7f4938681,The Doghouse Pub & Grill,33.590504,-86.163333,0.0,0.083964,x
413239,aa5ac3421388b6b3c67f7a545fd46cc9,JP BP - Palmetto,33.528677,-84.662145,8.0,c063c0736a23234b286331c4df438050,JP BP - Palmetto 264,33.528594,-84.662293,0.0,0.011939,x
422017,aeaeb80c3d5feeb0e2c2c7b0938045c2,Pilot Travel Center #312,33.682548,-85.263639,0.0,dd1016d687d5960a8f279198a94d0cc5,"Newborn TruckStop, Tallapoosa GA",33.681568,-85.264077,125.0,0.083789,y
431399,b32e8760418e68f23c811a1cfd6bda78,AmBest,34.227798,-83.498901,165.0,c4f7bb32268a2d67ce744a6fc57042d6,Love's Travel Stop # 765,34.227399,-83.499387,0.0,0.045386,x


In [189]:
id_x = a[["pin id_x", "truckParkingSpotCount_x"]].drop_duplicates().copy()
id_y = a[["pin id_y", "truckParkingSpotCount_y"]].drop_duplicates().copy()
id_x.rename(columns={"pin id_x": "pin id", "truckParkingSpotCount_x": "truckParkingSpotCount"}, inplace=True)
id_y.rename(columns={"pin id_y": "pin id", "truckParkingSpotCount_y": "truckParkingSpotCount"}, inplace=True)
unique_id = pd.concat([id_x, id_y], ignore_index=True)

In [190]:
unique_id.drop_duplicates(inplace=True)
unique_id.rename(columns={"truckParkingSpotCount": "ucount3"}, inplace=True)

In [191]:
a = unique_id.groupby(["pin id"]).agg({"ucount3": "count"}).reset_index()
unique_id[unique_id["pin id"].isin(a[a["ucount3"] > 1]["pin id"].unique())]

,pin id,ucount3


In [192]:
print(stop_tab_df.shape)
stop_tab_df = pd.merge(stop_tab_df, unique_id, on="pin id", how="left")
print(stop_tab_df.shape)
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["ucount3"].isnull(), stop_tab_df["truckParkingSpotCount"],
                                                stop_tab_df["ucount3"])

(789, 11)
(789, 12)


In [193]:
stop_tab_df["truckParkingSpotCount"] = np.where(stop_tab_df["pin id"].isin(list_to_rm), 0,
                                                stop_tab_df["truckParkingSpotCount"])

In [194]:
columns = ['pin id', 'pinname', 'lat', 'lng', 'truckParkingSpotCount', 'f_system',
           'link_id', 'review_score', 'amenities_score']

In [195]:
stop_tab_df[columns].to_excel("Model_Stops.xlsx", index=False)